# Encoding TEI with LLMs

This notebook demonstrates how to use Large Language Models (LLMs) to assist in the encoding of texts in the Text Encoding Initiative (TEI) format. The TEI is a standard for representing texts in digital form, and LLMs can help automate and enhance the encoding process.


In [21]:
pip install dotenv

Note: you may need to restart the kernel to use updated packages.


In [22]:
pip install openai

Note: you may need to restart the kernel to use updated packages.


In [23]:
pip install google-genai

Note: you may need to restart the kernel to use updated packages.


In [24]:
pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [25]:
import lxml

### Use GoogleAI API

Get a free API key from: https://aistudio.google.com/app/api-keys

Store your API key in a `.env` file as `GOOGLE_API_KEY=your_api_key`

14k queries limit per day:

* gemma-3-4b
* gemma-3-12b
* gemma-3-27b

20 queries limit per day:
* gemini-2.5-flash
* gemini-3-flash



In [26]:
from pathlib import Path
import os
from openai import OpenAI
from google import genai
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
## **** A CONFIGURER ****
provider = "openrouter" #"googleai" or "googleai"

models = ["openai/gpt-5-mini"] # valeurs possible pour OpenRouter: "openai/gpt-5-mini", "google/gemma-3-27b-it", "google/gemma-3-12b-it", ...
#models = ["gemma-3-27b-it"] # valeurs possibles pour GoogleAI : "gemma-3-4b-it", "gemma-3-12b-it", "gemma-3-27b-it"

In [27]:
SYSTEM_PROMPT = """
Tu es un expert en encodage XML TEI pour les dictionnaires anciens. Ton rôle est de transformer du texte OCR brut du *Dictionnaire de Trévoux* (1743) en un fichier XML structuré et valide.

### RÈGLES ABSOLUES
- **Sortie exclusive :** Ne génère QUE le code XML valide. AUCUN commentaire, aucune explication avant ou après.
- **Fidélité stricte :** Ne rien inventer, corriger, moderniser ou compléter. Ne pas ajouter du nouveau texte. Conserver la casse, la ponctuation et les fautes de l'OCR. Si c'est ambigu, encode tel quel.
- **Continuité :** Les entrées doivent suivre l'ordre exact du texte OCR. 
- **Fermeture :** Vérifie que chaque balise ouverte est fermée.

### STRUCTURE XML ATTENDUE
<TEI>
  <text>
    <body>
      <entry type="fragment" subtype="continued">
        <sense>
        Sens de l'entrée fragmentée.
        </sense>
      </entry>
      <entry type="mainEntry>
        <form type="lemmaGrp"><form type="lemma"><orth>LEMME</orth></form>
        <metamark function="lemmaDelimiter">, ou .</metamark>
        <gramGrp><gram>s.</gram> <gram>m.</gram></gramGrp></form> 
        <sense>
          Sens de l'entrée.
          <usg type="dom" ana="#nom_du_domaine">Terme de Domaine.</usg> 
        </sense>
        <entry type="relatedEntry>
        <form type="lemmaGrp"><form type="lemma"><orth>LEMME</orth></form>
        <metamark function="lemmaDelimiter">, ou .</metamark></form>
        <sense>
          Sens de la sous-entrée.
        </entry>       
      </entry>
    </body>
  </text>
</TEI>


### LOGIQUE D'ANALYSE ET D'ENCODAGE

**1. Gestion des entrées (`<entry>`)**
- **Nouvelle entrée :** Détectée par un mot/groupe de mots ENTIÈREMENT EN CAPITALES, situé en DÉBUT de paragraphe, suivi d'une ponctuation ("," ou ".") ou d'une marque grammaticale exprimée en abréviations. (Attention : un mot en majuscules au milieu d'une phrase n'est pas un lemme).
- **Fin de page :** Considère toujours l'entrée comme complète à la fin du texte fourni.
- **Cas Fragmentaire (Début de texte) :** Si le tout premier bloc du texte OCR ne commence pas par un lemme valide (ex: minuscule, phrase directe), c'est une continuation. Utilise `<entry type="fragment" subtype="continued">`. N'ajoute AUCUNE balise `<form>`.

**2. Lemme et Grammaire (`<orth>` et `<gram>`)**
- `<orth>` : Le premier segment en capitales s'il s'agit d'une entrée, ou seulement la première lettre en majuscule s'il s'agit d'une sous-entrée.
- `<gram>` : Il y a un `<gram>`pour chacune des abréviations grammaticales (ex: "s.", "m.", "v.", "act.") après le lemme.

**3. Sens et marque d'usage (`<sense>` et `<usg>`)**
- Crée un nouveau `<sense>` UNIQUEMENT s'il y a un marqueur explicite (une nouvelle entrée ou sous-entrée) ou une rupture sémantique très claire (par exemple : "un autre sens", "on le dit aussi de", "se dit aussi de"). Sinon, un seul `<sense>` global.
- `<usg>` contient de l'information sur l'utilisation du terme. Les `<usg>` ont un @type dont la valeur est "dom" (pour un usage qui dépend d'un domaine: "en termes de..."), 'time' (pour un usage qui dépend de la temporalité: "vieux mot"), "geo" (si l'usage dépend du lieu géographique), "style" (si l'usage dépend du style de la langue: "figure", "proverbe"). `<usg>`contient aussi un attribut @ana dont la valeur est le mot qui définit l'usage dans le texte (par exemple: <usg type="dom" ana="#fleuristerie")

**4. Une entrée principale (type="mainEntry") peut contneir une sous-entrée (type="relatedEntry")**
- La sous-entrée contient le même lemme que l'entrée, ou un lemme similaire.
- Le contenu de la sous-entrée s'organise comme celui de l'entrée.

**5. Les paragraphes (`<milestone unit="paragraph" rendition="#leftShift"/>`)**
- Un saut de ligne suppose un nouveau paragraphe, qu'il soit à l'intérieur d'un `<sense>`ou séparant deux `<sense>`différents.
- On indique le paragraphe avec `<milestone>` seulement s'il n'indique PAS le début d'une entrée, ni d'une sous-entrée.


### EXEMPLE 1
**Input 1 :** 
FABRIQUE, s. f. Maniére de construire quelque ouvrage. Fabrica, opus. La fabrique des draps d'Espagne est meilleure que celle de Hollande. On invente tous les jours de nouvelles fabriques d'étoffes. Il n'y a que des Officiers qui ont serment à Justice, qui osent travailler à la fabrique des monnoies.
Ce mot vient du Latin fabrica.
Fabrique, se dit aussi pour Construction d'un édifice ; mais il ne se dit guère qu'en parlant des Églises. Ce fonds est destiné pour la fabrique d'une Église Paroissiale. L'Ac. Ce mot se dit en Italie de tout bâtiment considérable. Il signifie aussi en François, la maniére de construire. Cet édifice est d'une belle fabrique. Toute la structure de la fabrique paroit riche. Daviler.

**Output 1 :**
  <entry type="mainEntry">
            <form type="lemmaGrp">
               <form type="lemma">
                  <orth>FABRIQUE</orth>
               </form>
               <metamark function="lemmaDelimiter">,</metamark>
               <gramGrp>
                  <gram type="pos" expand="substantif" norm="NOUN">s.</gram>
                  <gram type="gen" expand="feminin" norm="feminine">f.</gram>
               </gramGrp>
            </form>
            <sense>Maniére de construire quelque ouvrage. Fabrica, opus. La fabrique des draps
               d'Espagne est meilleure que celle de Hollande. On invente tous les jours de nouvelles
               fabriques d'étoffes. Il n'y a que des Officiers qui ont serment à Justice, qui osent
               travailler à la fabrique des monnoies.</sense>
            <milestone unit="paragraph" rendition="#leftShift"/>
            Ce mot vient du Latin fabrica.
            <entry type="relatedEntry">
               <form type="lemmaGrp">
                  <form type="lemma">
                     <orth>Fabrique</orth>
                  </form>
                  <metamark function="lemmaDelimiter">,</metamark>
               </form>
               <sense>se dit aussi pour Construction d'un édifice ; mais il ne se dit guère
                  qu'en parlant des Églises. Ce fonds est destiné pour la fabrique d'une Église
                  Paroissiale. L'Ac. Ce mot se dit en Italie de tout bâtiment considérable.</sense> <sense>Il
                  signifie aussi en François, la maniére de construire. Cet édifice est d'une belle
                  fabrique. Toute la structure de la fabrique paroit riche. Daviler.</sense>
            </entry>
    </entry>

### EXEMPLE 2
**Input 2 :**
PRÉSIDENTAL, ALE. adj. Ce qui concerne le Président. Praesidentalis. Voilà un homme qui affecte une gravité présidentale. Celui-là est savant en Droit, en Pratique, il a toutes les qualités présidentales. La robe présidentale est différente de celle des autres Magistrats.
PRÉSIDENTE. Terme de Fleuriste. Tulipe de couleur de rose tirant sur l'incarnat &amp; blanc d'entrée. Morin.

**Output 2 :**
<entry type="mainEntry">
            <form type="lemmaGrp">
               <form type="lemma"><orth>PRÉSIDENTAL</orth></form>
               <pc>,</pc>
               <form type="inflected">
                  <orth>ALE</orth>
               </form>
               <metamark function="lemmaDelimiter">.</metamark>
               <gramGrp>
                  <gram type="pos" expand="adjectif" norm="ADJ">adj.</gram>
               </gramGrp>
            </form>
            <sense>Ce qui concerne le Président. Praesidentalis. Voilà un homme qui affecte
               une gravité présidentale. Celui-là est savant en Droit, en Pratique, il a toutes les
               qualités présidentales. La robe présidentale est différente de celle des autres
               Magistrats.</sense>
         </entry>
         <entry type="mainEntry">
            <form type="lemmaGrp">
               <form type="lemma">
                  <orth>PRÉSIDENTE</orth>
               </form>
               <metamark function="lemmaDelimiter">.</metamark>
            </form>
            <sense>
               <usg type="dom" ana="#fleuriste">Terme de Fleuriste.</usg> Tulipe de couleur de
               rose tirant sur l'incarnat &amp; blanc d'entrée. Morin. </sense>
         </entry>

### EXEMPLE 3
**Input 3 :**
PREVÔT. s. m. Juge inférieur. Tribunus, Praepositus. Les Prevôts sont les premiers Juges Royaux, &amp; qui jugent les affaires civiles en première instance.
 Prevôt de l'armée. Officier préposé pour avoir l'inspection sur les délits qui se commettent dans l'armée par les soldats. On appelle aussi Prevôt dans quelques Régimens, l'Officier qui a pareille inspection sur les délits qui se commettent dans ces Régimens par les soldats ; &amp; Prevôt des Bandes, l'Officier qui a pareille jurisdiction dans le Régiment des Gardes. On a mis ce soldat entre les mains du Prevôt des Bandes.

**Output 3 :**
<entry type="mainEntry">
            <form type="lemmaGrp">
               <form type="lemma">
                  <orth>PREVÔT</orth>
               </form>
               <metamark function="lemmaDelimiter">.</metamark> 
               <gramGrp>
                  <gram type="pos" expand="substantif" norm="NOUN">s.</gram> 
                  <gram type="gen" expand="masculin" norm="masculine">m.</gram>
               </gramGrp>
            </form> 
            <sense>Juge inférieur. Tribunus, Praepositus. Les Prevôts sont les premiers
         Juges Royaux, &amp; qui jugent les affaires civiles en première instance.</sense> 
            <pb n="506"/>
            <entry type="relatedEntry">
               <form type="lemmaGrp">
                  <form type="compound">
                     <orth>Prevôt de l'armée</orth>
                  </form>
                  <metamark function="lemmaDelimiter">.</metamark>
               </form> 
               <sense>Officier préposé
         pour avoir l'inspection sur les délits qui se commettent dans l'armée par les soldats. On
         appelle aussi Prevôt dans quelques Régimens, l'Officier qui a pareille inspection sur les
         délits qui se commettent dans ces Régimens par les soldats ; &amp; Prevôt des Bandes,
         l'Officier qui a pareille jurisdiction dans le Régiment des Gardes. On a mis ce soldat
         entre les mains du Prevôt des Bandes.</sense></entry></entry>

### EXEMPLE 4
**Input 4 :**
LAZARET, s. m. C'est un bâtiment public fait en forme d'Hôpital, pour recevoir les pauvres, les pestiférés. Xenodochium, nosocomium suburbicanum. Il est destiné en quelques endroits à faire la quarantaine par ceux qui viennent des lieux suspects de peste. C'est une grande maison hors de la ville, dont les bâtimens sont séparés &amp; isolés, &amp; où l'équipage des vaisseaux demeure quarante jours, plus ou moins, selon les temps, &amp; l'endroit du départ. Le Lazaret de Milan est un des plus beaux Hôpitaux d'Italie.
On a appellé Lazares, les ladres ou lépreux, à cause que leur maison, ou Église, qui étoit hors des murs de Jérusalem, étoit dédiée à Saint Lazare.

**Output 4 :**
<entry type="mainEntry">
            <form type="lemmaGrp">
               <form type="lemma">
                  <orth>LAZARET</orth>
               </form>
               <metamark function="lemmaDelimiter">,</metamark>
               <gramGrp>
                  <gram type="pos" expand="substantif" norm="NOUN">s.</gram>
                  <gram type="gen" expand="masculin" norm="masculine">m.</gram>
               </gramGrp>
            </form>
            <sense>C'est un bâtiment public fait en forme d'Hôpital, pour recevoir les pauvres, les
               pestiférés. Xenodochium, nosocomium suburbicanum. Il est destiné en quelques endroits
               à faire la quarantaine par ceux qui viennent des lieux suspects de peste. C'est une
               grande maison hors de la ville, dont les bâtimens sont séparés &amp; isolés, &amp; où
               l'équipage des vaisseaux demeure quarante jours, plus ou moins, selon les temps,
               &amp; l'endroit du départ. Le Lazaret de Milan est un des plus beaux Hôpitaux
               d'Italie. <milestone unit="paragraph" rendition="#leftShift"/>On a appellé Lazares,
               les ladres ou lépreux, à cause que leur maison, ou Église, qui étoit hors des murs de
               Jérusalem, étoit dédiée à Saint Lazare.</sense></entry>
"""


def build_user_prompt(text: str) -> str:
    return f"""TEXTE À ENCODER :
{text}

TEXTE ENCODE EN XML TEI :
"""


def openrouter(client, model_name, pre_prompt, text):
    user_prompt = build_user_prompt(text)
    # Gemma → pas de system message
    if "gemma" in model_name.lower() or "mistral" in model_name.lower():

        full_prompt = pre_prompt + "\n\n" + user_prompt

        response = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "user", "content": full_prompt},
            ],
        )
    # Autres modèles → messages normaux
    else:
        response = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "system", "content": pre_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
    return response.choices[0].message.content.strip()


def googleai(client, model_name, pre_prompt, text):
    user_prompt = build_user_prompt(text)
    response = client.models.generate_content(
        model=model_name,
        contents=pre_prompt + "\n\n" + user_prompt
    )
    return response.text


def tei_encoding(pre_prompt, text, provider, client, model_name):
    try:
        if provider == "openrouter":
            return openrouter(client, model_name, pre_prompt, text)
        elif provider == "googleai":
            return googleai(client, model_name, pre_prompt, text)
        else:
            raise ValueError("Provider inconnu")

    except Exception as e:
        print(f"Erreur {provider}: {e}")
        return text

In [28]:
if provider == "googleai":
    client = genai.Client(
        api_key=os.getenv("GOOGLE_API_KEY")
        )
elif provider == "openrouter":
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

In [29]:

## **** A CONFIGURER ****
provider = "openrouter" #"googleai" or "googleai"

models = ["openai/gpt-5-mini", "google/gemma-3-27b-it"] # valeurs possible pour OpenRouter: "openai/gpt-5-mini", "google/gemma-3-27b-it", "google/gemma-3-12b-it", ...
#models = ["gemma-3-27b-it"] # valeurs possibles pour GoogleAI : "gemma-3-4b-it", "gemma-3-12b-it", "gemma-3-27b-it"

#ocr_path_name = "data/glm-ocr/txt/"
ocr_path_name = "./txt/input"
ocr_files = sorted(Path(ocr_path_name).glob("*.txt"))

prompts = sorted(Path("./prompts/").glob("*.txt"))

for prompt in prompts:
    with open(prompt) as f:
        pre_prompt = f.read()

    print("* "+prompt.stem)

    for model_name in models:
        if provider == "openrouter":
            model_short = model_name.split("/")[1].split(":")[0]
        elif provider == "googleai":
            model_short = model_name

        print("** "+ model_short)
        # create folder with model name if it doesn't exist
        output_path = Path(f"./output/{model_short}")
        output_path.mkdir(parents=True, exist_ok=True)

        output_path = Path(f"./output/{model_short}/{prompt.stem}")
        output_path.mkdir(parents=True, exist_ok=True)

        for ocr_file in tqdm(ocr_files):
            ocr_text = ocr_file.read_text(encoding="utf-8", errors="replace")

            # if file already exists, skip
            output_file = output_path / f"{ocr_file.stem}.xml"
            if output_file.exists():
                continue

            if provider == "openrouter":
                corrected_text = tei_encoding(pre_prompt, ocr_text, "openrouter", client, model_name)
            elif provider == "googleai":
                corrected_text = tei_encoding(pre_prompt, ocr_text, "googleai", client, model_name)
                # remove ```xml at the beginning and ``` at the end if they exist
                if corrected_text.startswith("```xml"):
                    corrected_text = corrected_text[len("```xml"):].strip()
                if corrected_text.endswith("```"):
                    corrected_text = corrected_text[:-len("```")].strip()

            # Write corrected text to output file
            output_file.write_text(corrected_text, encoding="utf-8", errors="replace")


* Tentative_1
** gpt-5-mini


100%|██████████| 10/10 [21:09<00:00, 126.91s/it]


** gemma-3-27b-it


100%|██████████| 10/10 [16:37<00:00, 99.79s/it] 


* Tentative_2
** gpt-5-mini


...

100%|██████████| 10/10 [20:06<00:00, 120.67s/it]


* Tentative_3
** gpt-5-mini


100%|██████████| 10/10 [14:36<00:00, 87.68s/it]


** gemma-3-27b-it


100%|██████████| 10/10 [20:33<00:00, 123.35s/it]


In [ ]:
# =====================================================
# USAGE (example de test rapide)
# =====================================================

text = "Ceci est un eximp1e de texte OQR avec des emeurs."

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

model_name = "google/gemma-3-27b-it"

print("OpenRouter (gemma):", tei_encoding(text, "openrouter", openrouter_client, model_name))